## Import Modules and dataset path

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
# Load data
RUN_ID = "6f701059-94c7-4969-8ca9-5ba2656995d9"
data_path = Path("../data/interim") / RUN_ID / f"validated_dataset_{RUN_ID}.parquet"

df = pd.read_parquet(data_path)

## Data Overview

In [ ]:
df.sample(3)

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Memory MB:", df.memory_usage(deep=True).sum() / 1e6)

In [ ]:
sorted(df.columns)

In [ ]:
df.info()

In [ ]:
numeric_features = df.select_dtypes(include=np.number).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()
datetime_features = df.select_dtypes(include=["datetime"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Datetime features:", len(datetime_features))

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.describe()

In [ ]:
df.id.duplicated().sum()

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes,
    "type":df.map(type).nunique(),
    "missing": df.isnull().sum(),
    "missing %": df.isnull().mean() * 100,
    "unique": df.nunique(),
    "cardinality %": df.nunique() / len(df) * 100
})

overview.sort_values("missing %", ascending=False)

## Statistical Profiling

In [ ]:
num_cols = df.select_dtypes(include=np.number)

dist_stats = pd.DataFrame({
    "skewness": num_cols.skew(),
    "kurtosis": num_cols.kurt(),
})

dist_stats

### Outliers

In [ ]:
Q1 = num_cols.quantile(0.25)
Q3 = num_cols.quantile(0.75)
IQR = Q3 - Q1

outliers = ((num_cols < (Q1 - 1.5 * IQR)) | (num_cols > (Q3 + 1.5 * IQR))).sum()

outliers_summary = pd.DataFrame({
    "outliers": outliers,
    "outlier_ratio": outliers / len(num_cols),
})

outliers_summary

## EDA & Visualization

#### Missing values

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
missing.head(20)

In [ ]:
ax = missing.plot(kind="bar", figsize=(20, 5))

ax.set_xlabel("Feature")
ax.set_ylabel("Missing Ratio")
ax.set_title("Missing Values by Feature")

#### Grade column

In [ ]:
grades = df.grade.value_counts(normalize=True)
grades

In [ ]:
grades.plot(kind="bar")

#### Target column

In [ ]:
df.loan_status.value_counts()

#### Loans per month

In [ ]:
(df.issue_d.min()), (df.issue_d.max())

In [ ]:
issue_year = df.issue_d.dt.year.value_counts().sort_index()
issue_year

In [ ]:
issue_year.plot(kind="bar")

In [ ]:
loans_per_month = df.groupby(df.issue_d.dt.to_period("M")).size()
loans_per_month

In [ ]:
loans_per_month.plot(figsize=(12, 5))

In [ ]:
# Boxplot of annual income
sns.boxplot(x=df.annual_inc)

#### Heatmap

In [ ]:
num_cols = df.select_dtypes(include="number").columns

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    mask=mask,
    cmap="coolwarm",
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()